### Installs

In [1]:
!pip install tabulate


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


### Imports

In [2]:
import numpy as np
import time
import os
import shutil
from pathlib import Path
from PIL import Image
from tabulate import tabulate

In [3]:
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import absl.logging
absl.logging.set_verbosity(absl.logging.ERROR)

In [4]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, regularizers, callbacks
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from scipy.ndimage import gaussian_filter

I0000 00:00:1779035703.003497    1554 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779035705.376934    1554 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [5]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [6]:
print("TensorFlow version:", tf.__version__)
print("GPUs available:", tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.21.0
GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


### Dataset Setup

In [7]:
IMG_H, IMG_W = 288, 96
INPUT_SHAPE = (IMG_H, IMG_W, 3)
BATCH = 32

In [8]:
RAW_DATA_DIR = Path('KolektorSDDv2')
ORGANIZED_DATA_DIR = Path('KolektorSDD2_organized')

In [9]:
def organize_split(raw_split_dir, organized_split_dir):
    (organized_split_dir / 'non_damaged').mkdir(parents=True, exist_ok=True)
    (organized_split_dir / 'damaged').mkdir(parents=True, exist_ok=True)
    image_paths = sorted(p for p in raw_split_dir.glob('*.png') if not p.stem.endswith('_GT'))
    for img_path in image_paths:
        mask_path = img_path.parent / (img_path.stem + '_GT.png')
        if not mask_path.exists():
            continue
        mask = np.array(Image.open(mask_path))
        target_class = 'damaged' if (mask > 0).any() else 'non_damaged'
        shutil.copy2(img_path, organized_split_dir / target_class / img_path.name)

In [10]:
if not ORGANIZED_DATA_DIR.exists():
    print(f"Organizing dataset from {RAW_DATA_DIR} → {ORGANIZED_DATA_DIR}")
    organize_split(RAW_DATA_DIR / 'train', ORGANIZED_DATA_DIR / 'train')
    organize_split(RAW_DATA_DIR / 'test',  ORGANIZED_DATA_DIR / 'test')

In [11]:
TRAIN_DIR = ORGANIZED_DATA_DIR / 'train'
TEST_DIR  = ORGANIZED_DATA_DIR / 'test'

In [12]:
def load_images(folder):
    paths = sorted(p for p in folder.glob('*.png') if not p.stem.endswith('_GT'))
    images = []
    for p in paths:
        img = Image.open(p).convert('RGB').resize((IMG_W, IMG_H))
        images.append(np.array(img, dtype=np.float32) / 255.0)
    return np.array(images)

In [13]:
X_train_normal  = load_images(TRAIN_DIR / 'non_damaged')
X_train_damaged = load_images(TRAIN_DIR / 'damaged')
X_test_normal   = load_images(TEST_DIR / 'non_damaged')
X_test_damaged  = load_images(TEST_DIR / 'damaged')

In [14]:
X_test = np.concatenate([X_test_normal, X_test_damaged])
y_test = np.array([0]*len(X_test_normal) + [1]*len(X_test_damaged))

In [15]:
X_all_train = np.concatenate([X_train_normal, X_train_damaged])
y_all_train = np.array([0]*len(X_train_normal) + [1]*len(X_train_damaged))

In [16]:
X_train_sup, X_val_sup, y_train_sup, y_val_sup = train_test_split(
    X_all_train, y_all_train,
    test_size=0.20, random_state=SEED, stratify=y_all_train
)

In [17]:
X_train_unsup = X_train_sup[y_train_sup == 0]
X_val_unsup   = X_val_sup[y_val_sup == 0]

In [18]:
n_test = len(X_test)
print(f"\nData loaded: {len(X_all_train)} train, {n_test} test")


Data loaded: 2331 train, 1004 test


### 1. PCA (39 components, var_threshold=0.30)

In [19]:
n_features = IMG_H * IMG_W * 3
X_train_flat = X_train_unsup.reshape(len(X_train_unsup), n_features)
X_test_flat  = X_test.reshape(n_test, n_features)

In [20]:
pca_mean = X_train_flat.mean(axis=0)
pca_std  = X_train_flat.std(axis=0) + 1e-8
X_train_centered = (X_train_flat - pca_mean) / pca_std
X_test_centered  = (X_test_flat - pca_mean) / pca_std

In [21]:
print("Fitting PCA (var_threshold=0.30)...")
t0 = time.perf_counter()
pca_final = PCA(n_components=0.30, random_state=SEED)
pca_final.fit(X_train_centered)
pca_train_time = time.perf_counter() - t0
print(f"  K={pca_final.n_components_} components, train time: {pca_train_time:.1f}s")

Fitting PCA (var_threshold=0.30)...
  K=39 components, train time: 113.2s


In [22]:
def pca_inference(X_centered):
    Z = pca_final.transform(X_centered)
    X_hat = pca_final.inverse_transform(Z)
    return ((X_centered - X_hat) ** 2).mean(axis=1)

### 2. CAE (latent=6)

In [23]:
SIGMA = 4

In [24]:
def build_cae(latent_channels=6):
    inp = layers.Input(shape=INPUT_SHAPE)
    x = inp
    for filters in [32, 64, 128, 256]:
        x = layers.Conv2D(filters, 3, padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(latent_channels, 1, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    for filters in [256, 128, 64, 32]:
        x = layers.Conv2DTranspose(filters, 3, strides=2, padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
    out = layers.Conv2D(3, 1, padding='same', activation='sigmoid')(x)
    return keras.Model(inp, out, name=f'cae_latent{latent_channels}')

In [25]:
def cae_inference(model, images):
    recon = model.predict(images, batch_size=32, verbose=0)
    err = ((images - recon) ** 2).mean(axis=-1)
    return np.array([float(gaussian_filter(err[i], sigma=SIGMA).max()) for i in range(len(err))])

In [26]:
tf.keras.utils.set_random_seed(SEED)
cae_model = build_cae(latent_channels=6)
cae_model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss='mse')

In [27]:
print("\nTraining CAE (latent=6)...")
t0 = time.perf_counter()
cae_model.fit(
    X_train_unsup, X_train_unsup,
    validation_data=(X_val_unsup, X_val_unsup),
    epochs=100, batch_size=32,
    callbacks=[callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=0)],
    verbose=0
)
cae_train_time = time.perf_counter() - t0
print(f"  Params: {cae_model.count_params():,}, train time: {cae_train_time:.1f}s")


Training CAE (latent=6)...
  Params: 795,297, train time: 165.9s


### 3. CNN v2 tuned (128 filters, lr=1e-4, l2=1e-5, dropout=0.3)

In [28]:
def build_cnn():
    reg = regularizers.l2(1e-5)
    return models.Sequential([
        layers.Input(shape=INPUT_SHAPE),
        layers.Conv2D(128, 3, padding='same', activation='relu', kernel_regularizer=reg),
        layers.MaxPooling2D(2),
        layers.Conv2D(256, 3, padding='same', activation='relu', kernel_regularizer=reg),
        layers.MaxPooling2D(2),
        layers.Conv2D(512, 3, padding='same', activation='relu', kernel_regularizer=reg),
        layers.GlobalMaxPooling2D(),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ])

In [29]:
n_neg = int((y_train_sup == 0).sum())
n_pos = int((y_train_sup == 1).sum())
class_weight = {0: 1.0, 1: n_neg / n_pos}

In [30]:
tf.keras.utils.set_random_seed(SEED)
cnn_model = build_cnn()
cnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=[keras.metrics.AUC(name='ap', curve='PR')]
)

In [31]:
print("\nTraining CNN v2_tuned...")
t0 = time.perf_counter()
cnn_model.fit(
    X_train_sup, y_train_sup,
    validation_data=(X_val_sup, y_val_sup),
    epochs=100, batch_size=32,
    class_weight=class_weight,
    callbacks=[callbacks.EarlyStopping(monitor='val_ap', mode='max', patience=8, restore_best_weights=True, verbose=0)],
    verbose=0
)
cnn_train_time = time.perf_counter() - t0
print(f"  Params: {cnn_model.count_params():,}, train time: {cnn_train_time:.1f}s")


Training CNN v2_tuned...
  Params: 1,479,425, train time: 256.1s


### 4. ResNet50 (frozen backbone + classification head)

In [32]:
def build_resnet():
    base = ResNet50(weights='imagenet', include_top=False, input_shape=INPUT_SHAPE)
    base.trainable = False
    inputs = keras.Input(shape=INPUT_SHAPE)
    x = preprocess_input(inputs * 255.0)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    model = keras.Model(inputs, outputs, name='resnet50_transfer')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=3e-4),
        loss='binary_crossentropy',
        metrics=[keras.metrics.AUC(name='ap', curve='PR')]
    )
    return model

In [33]:
tf.keras.utils.set_random_seed(SEED)
resnet_model = build_resnet()

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


In [34]:
print("\nTraining ResNet50...")
t0 = time.perf_counter()
resnet_model.fit(
    X_train_sup, y_train_sup,
    validation_data=(X_val_sup, y_val_sup),
    epochs=50, batch_size=32,
    class_weight=class_weight,
    callbacks=[callbacks.EarlyStopping(monitor='val_ap', mode='max', patience=10, restore_best_weights=True, verbose=0)],
    verbose=0
)
resnet_train_time = time.perf_counter() - t0
print(f"  Params: {resnet_model.count_params():,}, train time: {resnet_train_time:.1f}s")


Training ResNet50...
  Params: 23,850,113, train time: 48.0s


══════════════════════════════════════════════════════════════════════════════
 INFERENCE RUNTIME BENCHMARK
══════════════════════════════════════════════════════════════════════════════

In [35]:
N_WARMUP  = 3
N_REPEATS = 10

In [42]:
print(f"  Inference Benchmark: {N_WARMUP} warmup + {N_REPEATS} timed runs")
print(f"  Test set: {n_test} images")

  Inference Benchmark: 3 warmup + 10 timed runs
  Test set: 1004 images


In [37]:
def benchmark(name, fn):
    for _ in range(N_WARMUP):
        fn()
    times = []
    for _ in range(N_REPEATS):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    mean_t = np.mean(times)
    std_t  = np.std(times)
    print(f"  {name}: {mean_t:.4f} ± {std_t:.4f}s total, {mean_t/n_test*1000:.3f} ms/image")
    return mean_t, std_t

In [38]:
pca_inf_mean, pca_inf_std       = benchmark("PCA",      lambda: pca_inference(X_test_centered))
cae_inf_mean, cae_inf_std       = benchmark("CAE",      lambda: cae_inference(cae_model, X_test))
cnn_inf_mean, cnn_inf_std       = benchmark("CNN",      lambda: cnn_model.predict(X_test, batch_size=32, verbose=0))
resnet_inf_mean, resnet_inf_std = benchmark("ResNet50",  lambda: resnet_model.predict(X_test, batch_size=32, verbose=0))

  PCA: 0.4014 ± 0.0089s total, 0.400 ms/image
  CAE: 1.2360 ± 0.0295s total, 1.231 ms/image
  CNN: 0.4755 ± 0.0192s total, 0.474 ms/image
  ResNet50: 0.5270 ± 0.0204s total, 0.525 ms/image


── Final Table ───────────────────────────────────────────────────────────────

In [39]:
rows = [
    ["PCA",
     f"{pca_final.n_components_}",
     f"{pca_train_time:.1f}s",
     f"{pca_inf_mean/n_test*1000:.2f} ms"],

    ["CAE (latent=6)",
     f"{cae_model.count_params():,}",
     f"{cae_train_time:.1f}s",
     f"{cae_inf_mean/n_test*1000:.2f} ms"],

    ["CNN (v2 tuned)",
     f"{cnn_model.count_params():,}",
     f"{cnn_train_time:.1f}s",
     f"{cnn_inf_mean/n_test*1000:.2f} ms"],

    ["ResNet50",
     f"{resnet_model.count_params():,}",
     f"{resnet_train_time:.1f}s",
     f"{resnet_inf_mean/n_test*1000:.2f} ms"],
]

In [40]:
headers = ["Model", "Parameters", "Training Time", "Inference (per image)"]

In [45]:
print(tabulate(rows, headers=headers, tablefmt="fancy_grid"))

╒════════════════╤══════════════╤═════════════════╤═════════════════════════╕
│ Model          │   Parameters │ Training Time   │ Inference (per image)   │
╞════════════════╪══════════════╪═════════════════╪═════════════════════════╡
│ PCA            │           39 │ 113.2s          │ 0.40 ms                 │
├────────────────┼──────────────┼─────────────────┼─────────────────────────┤
│ CAE (latent=6) │      795,297 │ 165.9s          │ 1.23 ms                 │
├────────────────┼──────────────┼─────────────────┼─────────────────────────┤
│ CNN (v2 tuned) │    1,479,425 │ 256.1s          │ 0.47 ms                 │
├────────────────┼──────────────┼─────────────────┼─────────────────────────┤
│ ResNet50       │   23,850,113 │ 48.0s           │ 0.52 ms                 │
╘════════════════╧══════════════╧═════════════════╧═════════════════════════╛
